# 04 — Supervised Fine-Tuning (SFT) Baseline
**Goal**: Fine-tune the 1.3B base model on APPS problem-solution pairs using LoRA adapters. Produces the `./checkpoints/sft/final` adapter checkpoint to warm-start RL training (PPO & DPO).

---

## Step 1: Environment & Universal Path Resolution

In [1]:
import sys, os, shutil

# Universal Path Resolution & Auto-Copy for Kaggle / Local / Colab
def prepare_kaggle_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    # Search /kaggle/input for src folder
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(root, dest)
                print(f"Copied 'src' from {root} to {dest}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(src_dir, dest)
                print(f"Copied 'src' from {src_dir} to {dest}")
                return '/kaggle/working'
    
    # Fallback to parent directory
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src')):
        return parent
    return curr

repo_root = prepare_kaggle_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.sft import format_for_sft, run_sft_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

Copied 'src' from /kaggle/input/datasets/rajdeepbhowmick/self-correction-src/src to /kaggle/working/src
Project path added: /kaggle/working
Environment initialized!


## Step 2: Load APPS Dataset & Format for SFT

In [2]:
print("Loading APPS training dataset via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:2000]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)
print(f"Loaded {len(apps_clean)} APPS problem-solution pairs.")

sample_formatted = format_for_sft(apps_clean[0])
print("\n--- Sample Formatted SFT Prompt ---")
print(sample_formatted['text'][:300] + "...")


Loading APPS training dataset via Parquet branch...


all/train/0000.parquet:   0%|          | 0.00/45.0M [00:00<?, ?B/s]

competition/train/0000.parquet:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

interview/train/0000.parquet:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

introductory/train/0000.parquet:   0%|          | 0.00/8.21M [00:00<?, ?B/s]

all/test/0000.parquet:   0%|          | 0.00/424M [00:00<?, ?B/s]

... (4 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded 2000 APPS problem-solution pairs.

--- Sample Formatted SFT Prompt ---
### Problem:
Polycarp has $n$ different binary words. A word called binary if it contains only characters '0' and '1'. For example, these words are binary: "0001", "11", "0" and "0011100".

Polycarp wants to offer his set of $n$ binary words to play a game "words". In this game, players name words a...


## Step 3: Load Base Model & Run SFT Training

In [3]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading base model {MODEL_NAME} for SFT in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(model_name=MODEL_NAME, load_in_4bit=False, lora_r=16)

print("\nStarting SFT Training loop...")
trainer = run_sft_training(
    model=model,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir="./checkpoints/sft",
    num_epochs=1,
    per_device_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
)

print("\nSFT Training completed successfully!")
print("Saved fine-tuned adapter to ./checkpoints/sft/final")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Loading base model deepseek-ai/deepseek-coder-1.3b-instruct for SFT in FP16 precision...


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]


Starting SFT Training loop...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step,Training Loss
50,1.344850
100,1.270391
150,1.220787
200,1.181776
250,1.200734



SFT Training completed successfully!
Saved fine-tuned adapter to ./checkpoints/sft/final


In [4]:
import os
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Auto-search for checkpoint directory on Kaggle
checkpoint_dir = None
search_roots = ['./checkpoints/sft/final', '/kaggle/working/checkpoints/sft/final', '/kaggle/working', '/kaggle/input']

for root in search_roots:
    if os.path.exists(root):
        if os.path.exists(os.path.join(root, 'adapter_config.json')):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if 'adapter_config.json' in files:
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir:
    raise FileNotFoundError("Could not locate adapter_config.json in /kaggle/working or /kaggle/input")

print(f"=== Found Checkpoint Directory: {checkpoint_dir} ===")
for fname in sorted(os.listdir(checkpoint_dir)):
    fpath = os.path.join(checkpoint_dir, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  - {fname}: {size_mb:.2f} MB")

print("\n=== 2. Verifying LoRA Configuration ===")
config = PeftConfig.from_pretrained(checkpoint_dir)
print(f"  - lora_alpha: {config.lora_alpha}")
print(f"  - r: {config.r}")
print(f"  - target_modules: {list(config.target_modules)}")
print(f"  - peft_type: {config.peft_type}")

print("\n=== 3. Reloading Saved SFT Model & Adapter ===")
base_model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-1.3b-instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
reloaded_model = PeftModel.from_pretrained(base_model, checkpoint_dir)
reloaded_model.eval()
print("Reloaded SFT adapter model successfully!")

print("\n=== 4. Running APPS Inference Check (3 Examples) ===")
sample_prompts = [
    apps_clean[0]['question'],
    apps_clean[1]['question'],
    apps_clean[2]['question']
]

for idx, p in enumerate(sample_prompts):
    prompt_text = f"### Problem:\n{p[:300]}\n\n### Solution:\n```python\n"
    inputs = tokenizer(prompt_text, return_tensors="pt").to(reloaded_model.device)
    with torch.no_grad():
        out = reloaded_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\n--- Inference Sample {idx + 1} ---")
    print(gen_text[:250] + "...")

print("\nVerification completed successfully! SFT checkpoint is 100% functional.")

`torch_dtype` is deprecated! Use `dtype` instead!


=== Found Checkpoint Directory: ./checkpoints/sft/final ===
  - README.md: 0.00 MB
  - adapter_config.json: 0.00 MB
  - adapter_model.safetensors: 12.01 MB
  - chat_template.jinja: 0.00 MB
  - tokenizer.json: 2.18 MB
  - tokenizer_config.json: 0.00 MB

=== 2. Verifying LoRA Configuration ===
  - lora_alpha: 32
  - r: 16
  - target_modules: ['v_proj', 'q_proj']
  - peft_type: PeftType.LORA

=== 3. Reloading Saved SFT Model & Adapter ===


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


Reloaded SFT adapter model successfully!

=== 4. Running APPS Inference Check (3 Examples) ===


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.



--- Inference Sample 1 ---
### Problem:
Polycarp has $n$ different binary words. A word called binary if it contains only characters '0' and '1'. For example, these words are binary: "0001", "11", "0" and "0011100".

Polycarp wants to offer his set of $n$ binary words to play ...


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.



--- Inference Sample 2 ---
### Problem:
Mikhail walks on a Cartesian plane. He starts at the point $(0, 0)$, and in one move he can go to any of eight adjacent points. For example, if Mikhail is currently at the point $(0, 0)$, he can go to any of the following points in one m...

--- Inference Sample 3 ---
### Problem:
You are given three sequences: $a_1, a_2, \ldots, a_n$; $b_1, b_2, \ldots, b_n$; $c_1, c_2, \ldots, c_n$.

For each $i$, $a_i \neq b_i$, $a_i \neq c_i$, $b_i \neq c_i$.

Find a sequence $p_1, p_2, \ldots, p_n$, that satisfy the following...

Verification completed successfully! SFT checkpoint is 100% functional.
